# 08 · Linguistics, sequences and graph NLP

Additional syllabus lab: morphology, syntax, collocation, POS/NER/dependencies, hidden-state sequences and graph-based term ranking. These are teaching demonstrations, not extra validated production models.

In [1]:
from pathlib import Path
import sys, json, os
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists(): ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), 'Open this notebook from CineScope/notebooks or CineScope.'
sys.path.insert(0, str(ROOT / 'backend'))
sys.path.insert(0, str(ROOT / 'scripts'))
os.chdir(ROOT)
(ROOT / 'artifacts').mkdir(exist_ok=True)
SEED = 42

In [2]:
from collections import Counter,defaultdict
import math,re,numpy as np
sentences=['The brilliant actor tells a moving story','The actor gives a brilliant performance','A moving story needs good pacing']
unigrams=Counter();bigrams=Counter()
for sentence in sentences:
    tokens=sentence.lower().split();unigrams.update(tokens);bigrams.update(zip(tokens,tokens[1:]))
total=sum(unigrams.values())
pmi={pair:math.log2((count/sum(bigrams.values()))/((unigrams[pair[0]]/total)*(unigrams[pair[1]]/total))) for pair,count in bigrams.items()}
print('Illustrative PMI collocations:',sorted(pmi.items(),key=lambda x:-x[1])[:6])
print('Tiny-corpus PMI overweights rare pairs; require a frequency floor on real data.')

Illustrative PMI collocations: [(('needs', 'good'), 4.495855026887171), (('good', 'pacing'), 4.495855026887171), (('actor', 'tells'), 3.4958550268871713), (('moving', 'story'), 3.4958550268871713), (('actor', 'gives'), 3.4958550268871713), (('brilliant', 'performance'), 3.4958550268871713)]
Tiny-corpus PMI overweights rare pairs; require a frequency floor on real data.


In [3]:
try:
    import spacy
    nlp=spacy.load('en_core_web_sm')
    doc=nlp('The excellent actor in Interstellar gives a moving performance.')
    display([(t.text,t.lemma_,t.pos_,t.dep_,t.head.text) for t in doc])
    print('Entities:',[(e.text,e.label_) for e in doc.ents])
except (ImportError,OSError):
    print('Optional: install transformers extras and run python -m spacy download en_core_web_sm for morphology/POS/NER/dependency parsing.')

Optional: install transformers extras and run python -m spacy download en_core_web_sm for morphology/POS/NER/dependency parsing.


## Hidden Markov Model and Viterbi
A hidden sequence (e.g. POS tags) emits observed words. Viterbi finds the most likely tag path from transition and emission probabilities. A CRF instead models the conditional label sequence and can incorporate contextual features; neither should be confused with a transformer language model.

In [4]:
def viterbi(observations,states,start,transition,emission,unknown=1e-6):
    log=lambda p:math.log(max(p,1e-12))
    scores={s:log(start[s])+log(emission[s].get(observations[0],unknown)) for s in states}
    paths={s:[s] for s in states}
    for observation in observations[1:]:
        new_scores={};new_paths={}
        for state in states:
            prev=max(states,key=lambda p:scores[p]+log(transition[p][state]))
            new_scores[state]=scores[prev]+log(transition[prev][state])+log(emission[state].get(observation,unknown))
            new_paths[state]=paths[prev]+[state]
        scores,paths=new_scores,new_paths
    return paths[max(scores,key=scores.get)]
states=['ADJ','NOUN'];start={'ADJ':.8,'NOUN':.2};transition={'ADJ':{'ADJ':.2,'NOUN':.8},'NOUN':{'ADJ':.6,'NOUN':.4}}
emission={'ADJ':{'great':.6,'slow':.4},'NOUN':{'acting':.5,'pacing':.5}}
print(viterbi(['great','acting'],states,start,transition,emission))

['ADJ', 'NOUN']


In [5]:
# Graph NLP: word co-occurrence graph and PageRank-style centrality.
corpus=[s.lower().split() for s in sentences]
vocab=sorted(set(w for s in corpus for w in s));index={w:i for i,w in enumerate(vocab)}
adj=np.zeros((len(vocab),len(vocab)))
for sentence in corpus:
    for i,w in enumerate(sentence):
        for v in sentence[i+1:i+3]:
            adj[index[w],index[v]]+=1;adj[index[v],index[w]]+=1
row_sums=adj.sum(axis=1,keepdims=True)
transition=np.divide(adj,row_sums,out=np.full_like(adj,1/len(vocab)),where=row_sums!=0)
rank=np.full(len(vocab),1/len(vocab))
for _ in range(100):rank=.15/len(vocab)+.85*transition.T@rank
print(sorted(zip(vocab,rank),key=lambda x:-x[1])[:8])

[('a', np.float64(0.15595658382164845)), ('actor', np.float64(0.11051763524651143)), ('story', np.float64(0.10206926850184107)), ('brilliant', np.float64(0.0991527601040809)), ('moving', np.float64(0.09907780509462506)), ('needs', np.float64(0.07930836849776526)), ('the', np.float64(0.06775612748384474)), ('gives', np.float64(0.06762112629554974))]


## Extension choices
The project uses DistilBERT for classification, sentence-transformer embeddings for retrieval, and a T5-style encoder-decoder for optional grounded wording/summarization. GPT-style decoders, CRFs, machine translation and open-domain question answering are separate syllabus comparisons, not claimed implemented features of this movie app. Movie Q&A is grounded in the catalog and review summaries. Explain encoder-only vs encoder-decoder vs decoder-only architectures during the viva, including their sequence modeling objectives and evaluation requirements.